# 04 营销收益情景测算

本章使用假设购买率和成本参数研究何时可能划算，没有实际发券结果。

In [1]:
from pathlib import Path
import sys
# 兼容从项目根目录或 notebooks 目录打开。
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'run_analysis.py').exists() and (p / 'src/analysis.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image


In [2]:
from src.analysis import run_scenarios, estimate_profit, experiment_sample_size
users = pd.read_csv(ROOT/'outputs/improved/user_segments.csv')
import json
parameters = json.loads((ROOT/'configs/scenarios.json').read_text(encoding='utf-8'))
parameters

{'baseline_rate': 0.01,
 'activity_rates': [0.02, 0.05, 0.2],
 'margin': 0.4,
 'coupon': 10,
 'contact': 0.5,
 'currency': '原数据未明确；沿用统一假设金额单位',
 'assumptions': '每名购买用户一次订单、一次核销；客单价为优惠前金额，毛利率未扣券；同类型用户跨策略使用相同响应参数'}

## 收益口径
假设每位购买用户一次下单且核销一次；客单价为优惠前金额，毛利率未扣券。自然会购买的活动组用户也产生优惠成本。

```text
增量净收益 = 人数 × [(活动后购买率 − 自然购买率) × 客单价 × 毛利率
                     − 活动后购买率 × 券额 − 每人联系成本]
ROI = 增量净收益 / 营销成本
```
成本为 0 时 ROI 为未定义。

In [3]:
scenarios, sensitivity, strategies = run_scenarios(users)
scenarios[['情景','目标人数','活动后购买率','预计增量毛利','预计营销成本','预计增量净收益','ROI']].round(4)

,情景,目标人数,活动后购买率,预计增量毛利,预计营销成本,预计增量净收益,ROI
0,较低提升,89,0.02,37.0379,62.3,-25.2621,-0.4055
1,中等提升,89,0.05,148.1515,89.0,59.1515,0.6646
2,较高提升,89,0.20,703.7198,222.5,481.2198,2.1628


## 盈亏平衡
固定其他参数，求活动后购买率达到多少时净收益为零。这里的基准率、毛利率和成本是假设，不是历史因果估计。

In [4]:
aov = float(users.Average_Order_Value.mean())
p0, margin, coupon, contact = 0.01, 0.40, 10, 0.5
denominator = aov * margin - coupon
breakeven = (p0 * aov * margin + contact) / denominator if denominator > 0 else None
print('盈亏平衡购买率', f'{breakeven:.2%}' if breakeven is not None and breakeven <= 1 else '在 0–100% 范围内不可达')

盈亏平衡购买率 2.90%


## 券额与毛利率敏感性
这里固定活动后购买率 5%，比较不同成本参数。真实购买率也可能随券额变化，需要实验数据进一步识别。

In [5]:
sensitivity[['假设毛利率','每次核销优惠额','预计增量净收益','ROI']].round(4)

,假设毛利率,每次核销优惠额,预计增量净收益,ROI
0,0.2,5,7.3258,0.1097
1,0.2,10,-14.9242,-0.1677
2,0.2,15,-37.1742,-0.3342
3,0.4,5,81.4015,1.2195
4,0.4,10,59.1515,0.6646
5,0.4,15,36.9015,0.3317
6,0.6,5,155.4773,2.3292
7,0.6,10,133.2273,1.4969
8,0.6,15,110.9773,0.9975


## 两种人群选择方式的同口径测算
相同类型用户在两种策略中共享同样的响应参数。两种策略覆盖 200 人和 159 人，人数不同；即使净收益较高，也不能据此认定策略效果已被验证。

In [6]:
strategies.round(4)

,目标人数,预计增量销售额,预计增量毛利,预计优惠成本,预计联系成本,预计营销成本,预计增量净收益,ROI
策略,,,,,,,,
方案A_基础分前20%,200,1942.4081,776.9633,577.0,100.0,677.0,99.9633,0.1477
方案B_核心与潜力,159,2123.4360,849.3744,388.0,79.5,467.5,381.8744,0.8168


## A/B 计划所需规模
基准购买率 1%，预期 2%，双侧显著性水平 5%，功效 80%，1∶1 分流。下面是两独立比例的简化正态近似。

In [7]:
per_group = experiment_sample_size()
print('每组约', per_group, '人；合计约', per_group*2, '人')
print('当前符合资格的公开样本只有 40 人，不足以据此执行有效性验证。')

每组约 2320 人；合计约 4640 人
当前符合资格的公开样本只有 40 人，不足以据此执行有效性验证。


实验方案见 `reports/ab_experiment_design.md`，状态为待执行。